In [ ]:
import pandas as pd
import os
import numpy as np
import rdkit
from pyopsin import PyOpsin
from rdkit import Chem,DataStructs
from rdkit.Chem import Draw, Descriptors, Crippen, Lipinski, MolSurf
from rdkit.Chem import rdFingerprintGenerator

In [ ]:
for i in range( round( len( binding.index ) / chunk_size ) + 1 ):  
    
    chunk = pd.read_csv(f"chunks/chunk_{i}_{trial_num}.csv")

    print(i,' -- ', chunk['Ki (nM)'].isna().mean())

# Drug properties

In [ ]:
def bondFeatures(smiles):
    mol = Chem.MolFromSmiles(smiles)

    single = 0
    double = 0
    triple = 0
    aromatic = 0
    conjugated = 0
    total_bonds = mol.GetNumBonds()
    ringBonds = 0
    
    for bond in mol.GetBonds():
        if bond.IsInRing():
            ringBonds +=1
        btype = bond.GetBondType()
        if btype == Chem.rdchem.BondType.SINGLE:
            single += 1
        elif btype == Chem.rdchem.BondType.DOUBLE:
            double += 1
        elif btype == Chem.rdchem.BondType.TRIPLE:
            triple += 1
        elif btype == Chem.rdchem.BondType.AROMATIC:
            aromatic += 1
        
        if bond.GetIsConjugated():
            conjugated += 1


    return {
        "single_bonds": single,
        "double_bonds": double,
        "triple_bonds": triple,
        "aromatic_bonds": aromatic,
        "rotatable_bonds": Lipinski.NumRotatableBonds(mol),
        "conjugated_bonds": conjugated,
        "conjugation_ratio": conjugated / total_bonds if total_bonds else 0,
        "bonds_in_ring": ringBonds
    }
    
def hybridization(smiles):
    
    mol = Chem.MolFromSmiles(smiles)
    SP3 = 0
    SP2 = 0
    SP = 0
    
    for i in mol.GetAtoms():
        hybridization = i.GetHybridization()
        if hybridization == Chem.rdchem.HybridizationType.SP3:
            SP3 +=1
        elif hybridization == Chem.rdchem.HybridizationType.SP2:
            SP2 +=1
        elif hybridization == Chem.rdchem.HybridizationType.SP:
            SP +=1

    return {
        'SP3' : SP3,
        'SP2' : SP2,
        'SP' : SP
    }
    
def formalCharge(smiles):
    
    mol = Chem.MolFromSmiles(smiles)
    overall_charge = 0
    for i in mol.GetAtoms():
        overall_charge += i.GetFormalCharge()

    return overall_charge
    
def formalCharge(smiles):
    
    mol = Chem.MolFromSmiles(smiles)
    overall_charge = 0
    for i in mol.GetAtoms():
        overall_charge += i.GetFormalCharge()

    return overall_charge
    
def TPSA(smiles):
    mol = Chem.MolFromSmiles(smiles)
    Tpsa = MolSurf.TPSA(mol)
    return Tpsa

def TPSA(smiles):
    mol = Chem.MolFromSmiles(smiles)
    Tpsa = MolSurf.TPSA(mol)
    return Tpsa

def logP(smiles):
    mol = Chem.MolFromSmiles(smiles)
    logp = Crippen.MolLogP(mol)
    return logp

def Mr(smiles):
    mol = Chem.MolFromSmiles(smiles)
    weight = Descriptors.ExactMolWt(mol)
    return weight

def morganFingerprint(smiles):
    mol = Chem.MolFromSmiles(smiles)
    morgan_fp_gen = rdFingerprintGenerator.GetMorganGenerator(includeChirality=True, radius=2, fpSize=2048,countSimulation=False)
    
    fp = morgan_fp_gen.GetFingerprint(mol)
    arr = np.zeros((2048,), dtype=int)
    
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr